<a href="https://colab.research.google.com/github/Ankit1478/-CrackYourInternship-/blob/main/KV_cache.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch torchvision torchaudio


In [2]:
import torch

assert torch.cuda.is_available(), "GPU not enabled! Go to Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


GPU: Tesla T4
Total VRAM: 15.64 GB


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-135M"
device = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device)
model.eval()

print(f"Model loaded! VRAM used: {torch.cuda.memory_allocated() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded! VRAM used: 269.1 MB


In [10]:
prompt = "The river bank is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

print("Prompt tokens:", tokenizer.convert_ids_to_tokens(input_ids[0]))
print("input_ids shape:", tuple(input_ids.shape))  # (batch_size, seq_len)

with torch.no_grad():
    # Run ONLY Prefill (processes all prompt tokens in one parallel pass)
    outputs = model(input_ids=input_ids, use_cache=True)

# Grab the KV cache created during Prefill
kv_cache = outputs.past_key_values

# Inspect Layer 0's K and V tensors (compatible with transformers v4 & v5)
if hasattr(kv_cache, "layers"):
    k_layer0 = kv_cache.layers[0].keys
    v_layer0 = kv_cache.layers[0].values
else:
    k_layer0 = kv_cache.key_cache[0]
    v_layer0 = kv_cache.value_cache[0]

print("\n--- KV CACHE AFTER PREFILL ---")
print("Total layers cached:", len(kv_cache))
print("Layer 0 K shape:", tuple(k_layer0.shape))
print("Layer 0 V shape:", tuple(v_layer0.shape))
print("Stored on device:", k_layer0.device, "| dtype:", k_layer0.dtype)

# Peek at the 1st token the model wants to generate next
# for i in range(4):
#     guess_id = outputs.logits[:, i, :].argmax(dim=-1)
#     print(f"Index {i} guess:", repr(tokenizer.decode(guess_id)))
next_token_id = outputs.logits[:, -1, :].argmax(dim=-1)


Prompt tokens: ['The', 'Ġriver', 'Ġbank', 'Ġis']
input_ids shape: (1, 4)

--- KV CACHE AFTER PREFILL ---
Total layers cached: 30
Layer 0 K shape: (1, 3, 4, 64)
Layer 0 V shape: (1, 3, 4, 64)
Stored on device: cuda:0 | dtype: torch.float16


In [11]:
# 1. Reshape the predicted token (' a') to (1, 1) -> (batch=1, seq_len=1)
decode_input = next_token_id.view(1, 1)
print("Feeding into Decode:", repr(tokenizer.decode(decode_input[0])), "| shape:", tuple(decode_input.shape))

with torch.no_grad():
    # 2. Run 1 DECODE step: pass ONLY the 1 new token + the saved KV cache
    decode_outputs = model(
        input_ids=decode_input,
        past_key_values=kv_cache,
        use_cache=True
    )

# 3. Inspect the updated KV cache
kv_cache = decode_outputs.past_key_values

if hasattr(kv_cache, "layers"):
    k_layer0 = kv_cache.layers[0].keys
else:
    k_layer0 = kv_cache.key_cache[0]

print("\n--- KV CACHE AFTER 1 DECODE STEP ---")
print("Layer 0 K shape:", tuple(k_layer0.shape))

# 4. Pick the next predicted token
next_token_id = decode_outputs.logits[:, -1, :].argmax(dim=-1)
print("Next predicted token:", repr(tokenizer.decode(next_token_id)))

Feeding into Decode: ' a' | shape: (1, 1)

--- KV CACHE AFTER 1 DECODE STEP ---
Layer 0 K shape: (1, 3, 5, 64)
Next predicted token: ' very'


# **VLLM SETUP**

Cell 1: Load Model & vLLM Startup (Pre-allocate VRAM Pool)
Loads SmolLM2-135M and reserves the fixed kv_pool upfront

In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
model.eval()

# --- vLLM STARTUP: PRE-ALLOCATE KV POOL ---
NUM_LAYERS = model.config.num_hidden_layers       # 30
NUM_KV_HEADS = model.config.num_key_value_heads   # 3 (GQA)
HEAD_DIM = model.config.hidden_size // model.config.num_attention_heads  # 64
BLOCK_SIZE = 4      # 4 tokens per physical block
NUM_BLOCKS = 10     # 10 blocks total (max 40 tokens)

kv_pool = torch.empty(
    (NUM_LAYERS, 2, NUM_BLOCKS, NUM_KV_HEADS, BLOCK_SIZE, HEAD_DIM),
    dtype=torch.float16,
    device=device
)

free_blocks = list(range(NUM_BLOCKS))   # [0, 1, 2, ..., 9]
block_table = []                        # Logical block -> Physical block ID
num_cached_tokens = 0
all_tokens_text = []                    # For readable logs

pool_mb = (kv_pool.numel() * kv_pool.element_size()) / (1024 ** 2)
print(f"Loaded {model_id} on {device}")
print(f"KV Pool shape: {tuple(kv_pool.shape)} | Reserved VRAM: {pool_mb:.2f} MB")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loaded HuggingFaceTB/SmolLM2-135M on cuda
KV Pool shape: (30, 2, 10, 3, 4, 64) | Reserved VRAM: 0.88 MB


**Step 1: Read from Hugging Face (get_hf_kv)**

Job: Only extracts the K and V tensors for a specific layer from Hugging Face's cache (handles both older and newer transformers versions).

In [21]:
def get_hf_kv(hf_cache, layer_idx):
    if hasattr(hf_cache, "layers"):
        return hf_cache.layers[layer_idx].keys, hf_cache.layers[layer_idx].values
    return hf_cache.key_cache[layer_idx], hf_cache.value_cache[layer_idx]

Step 2: Find the Logical Address (get_block_and_slot)

In PagedAttention, tokens in a prompt are processed as a continuous sequence (`pos = 0, 1, 2, 3, 4...`), but GPU memory is divided into fixed-size pages called blocks (for example, `BLOCK_SIZE = 4` tokens per block). Step 2 (`get_block_and_slot`) maps each flat token position into a two-part logical address: the **logical block** (`token_pos // BLOCK_SIZE`) and the **slot** (`token_pos % BLOCK_SIZE`). Integer division (`//`) determines which sequential block the token belongs to, while the remainder (`%`) determines its exact seat inside that block. For instance, tokens `0` through `3` fill Logical Block `0` at slots `0` through `3`, and token `4` rolls over to Logical Block `1`, Slot `0`. This is called a *logical* address because it only represents the neat, ordered layout from the prompt's perspective—it does not know where the data actually lives in GPU VRAM. Translating that logical block into an actual physical block in `kv_pool` is handled separately by the Block Manager in Step 3.

In [22]:
def get_block_and_slot(token_pos, block_size=BLOCK_SIZE):
    logical_block = token_pos // block_size
    slot = token_pos % block_size
    return logical_block, slot

Step 3: Manage Physical VRAM Blocks (get_or_allocate_block)

In [ ]:
Job: The Block Manager. Checks if logical_block already has a physical block in block_table. If not, pops a fresh one from free_blocks.

In [23]:
def get_or_allocate_block(logical_block):
    if logical_block >= len(block_table):
        phys_block = free_blocks.pop(0)
        block_table.append(phys_block)
        return phys_block, "ALLOC"
    return block_table[logical_block], "REUSE"

Step 4: Write to kv_pool (write_kv_to_pool)
Job: Only copies 1 token's K and V across all 30 layers into the exact (phys_block, slot) in VRAM.

In [24]:
def write_kv_to_pool(hf_cache, token_pos, phys_block, slot):
    for layer_idx in range(NUM_LAYERS):
        k_hf, v_hf = get_hf_kv(hf_cache, layer_idx)
        kv_pool[layer_idx, 0, phys_block, :, slot, :] = k_hf[0, :, token_pos, :]
        kv_pool[layer_idx, 1, phys_block, :, slot, :] = v_hf[0, :, token_pos, :]

Putting Them Together (cache_token)
Now the main wrapper is just 5 readable lines that call each step in order:

In [26]:
def cache_token(hf_cache, token_pos, token_text):
    global num_cached_tokens
    logical_block, slot = get_block_and_slot(token_pos)
    phys_block, action = get_or_allocate_block(logical_block)
    write_kv_to_pool(hf_cache, token_pos, phys_block, slot)

    num_cached_tokens += 1
    all_tokens_text.append(token_text)
    bar = "█" * (slot + 1) + "░" * (BLOCK_SIZE - slot - 1)
    print(f"  pos={token_pos} | {token_text!r:10} | {action} -> Phys Block {phys_block} | Slot {slot} [{bar}] | block_table={block_table}")

Step5 Prefill Function

In [27]:
def run_prefill(prompt):
    print("\n" + "=" * 60)
    print(f"PREFILL: {prompt!r}")
    print("=" * 60)

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    # Compute K & V for all prompt tokens in one parallel forward pass
    with torch.no_grad():
        outputs = model(input_ids, use_cache=True)

    hf_cache = outputs.past_key_values
    next_token_id = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    # Write each prompt token into our paged kv_pool
    for pos, tid in enumerate(input_ids[0]):
        cache_token(hf_cache, pos, tokenizer.decode([tid]))

    unused_mb = pool_mb * (len(free_blocks) / NUM_BLOCKS)
    print(f"Prefill done -> Cached: {num_cached_tokens} tokens | Unused VRAM: {unused_mb:.2f} MB")
    return hf_cache, next_token_id

Step 6: Decode Function (run_decode)

In [37]:
def run_decode(hf_cache, next_token_id, max_new_tokens=5):
    print("\n" + "=" * 60)
    print(f"DECODE: Generating {max_new_tokens} tokens")
    print("=" * 60)

    for step in range(max_new_tokens):
        with torch.no_grad():
            decode_out = model(next_token_id, past_key_values=hf_cache, use_cache=True)

        hf_cache = decode_out.past_key_values
        token_text = tokenizer.decode([next_token_id[0][0]])

        # Write this 1 new token into our paged kv_pool
        cache_token(hf_cache, num_cached_tokens, token_text)

        # Pick the next token for the following decode step
        next_token_id = decode_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    unused_mb = pool_mb * (len(free_blocks) / NUM_BLOCKS)
    print(f"\nDecode done  -> Cached: {num_cached_tokens} tokens | Unused VRAM: {unused_mb:.2f} MB")
    print(f"Full output  -> {''.join(all_tokens_text)!r}")
    return hf_cache

Step 7: Run Both Together

In [34]:
kv_cache, next_token = run_prefill("Python is a great")


PREFILL: 'Python is a great'
  pos=0 | 'Python'   | REUSE -> Phys Block 0 | Slot 0 [█░░░] | block_table=[0, 1, 2, 3, 4]
  pos=1 | ' is'      | REUSE -> Phys Block 0 | Slot 1 [██░░] | block_table=[0, 1, 2, 3, 4]
  pos=2 | ' a'       | REUSE -> Phys Block 0 | Slot 2 [███░] | block_table=[0, 1, 2, 3, 4]
  pos=3 | ' great'   | REUSE -> Phys Block 0 | Slot 3 [████] | block_table=[0, 1, 2, 3, 4]
Prefill done -> Cached: 22 tokens | Unused VRAM: 0.44 MB


For every single token, we are adding its **Key ($K$)** and **Value ($V$)** vectors across **all 30 Transformer layers**.

Specifically, for **1 token** in your SmolLM2-135M setup:

* **Keys ($K$):** `30 layers × 3 KV heads × 64 numbers` (`FP16`) — written to `kv_pool[layer_idx, 0, phys_block, :, slot, :]`.
* **Values ($V$):** `30 layers × 3 KV heads × 64 numbers` (`FP16`) — written to `kv_pool[layer_idx, 1, phys_block, :, slot, :]`.

**What we are *not* adding:**

* **Queries ($Q$):** Never cached—a token's $Q$ is only used once at the moment it is generated to look back at past $K$s, then thrown away.
* **The text or token ID itself:** The KV cache only stores those floating-point $K$ and $V$ tensors.

We store them so that when the model generates the next token, it can read all previous tokens' $K$ and $V$ straight from `kv_pool` instead of recomputing the whole prompt from scratch.

In [39]:
def reset_kv_pool():
    global free_blocks, block_table, num_cached_tokens, all_tokens_text
    free_blocks = list(range(NUM_BLOCKS))
    block_table = []
    num_cached_tokens = 0
    all_tokens_text = []

# Always run all 3 lines together
reset_kv_pool()
kv_cache = run_decode(kv_cache, next_token, max_new_tokens=5)


DECODE: Generating 5 tokens
  pos=0 | ' way'     | ALLOC -> Phys Block 0 | Slot 0 [█░░░] | block_table=[0]
  pos=1 | ' to'      | REUSE -> Phys Block 0 | Slot 1 [██░░] | block_table=[0]
  pos=2 | ' learn'   | REUSE -> Phys Block 0 | Slot 2 [███░] | block_table=[0]
  pos=3 | ' Python'  | REUSE -> Phys Block 0 | Slot 3 [████] | block_table=[0]
  pos=4 | '.'        | ALLOC -> Phys Block 1 | Slot 0 [█░░░] | block_table=[0, 1]

Decode done  -> Cached: 5 tokens | Unused VRAM: 0.70 MB
Full output  -> ' way to learn Python.'
